In [1]:
import ee
import geemap

In [2]:
from tgbs_rs.config.config import (
    BASELINE_START, 
    BASELINE_END, 
    CURRENT_START, 
    CURRENT_END, 
    S2_BASELINE_START, 
    S2_ALL_BANDS, 
    HLS_MERGED_BANDS, 
    DRIVE_FOLDER
)
from tgbs_rs.config.config_vis import S2_VIS_PARAMS, L8_VIS_PARAMS, HLS_VIS_PARAMS
from tgbs_rs.preprocessing.hls_preprocessing import get_hls_merged_collection
from tgbs_rs.preprocessing.sentinel_preprocessing import get_s2_sr_collection
from tgbs_rs.utils import build_default_sites_featurecollection, get_sites_geometry
from tgbs_rs.summaries import build_period_composites, collection_to_site_timeseries
from tgbs_rs.export_rasters import export_table_to_drive

In [3]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project= "charrell-personal")

In [5]:
#Initialize Map
Map = geemap.Map()
Map = geemap.Map(height="800px")

In [4]:
# Build Site Feature Collection
sites_fc = build_default_sites_featurecollection()

#### Build HLS Annual and Monthly Composite Collections

In [17]:
# Build HLS Source Collection
hls_col = get_hls_merged_collection(
    aoi=sites_fc,
    start_date=ee.Date(BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    apply_water_masking=True
)

In [18]:
# Build HLS Annual Multiband Composite
hls_annual_col = build_period_composites(
    collection=hls_col,
    bands=HLS_MERGED_BANDS,
    start_date=ee.Date(BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    temporal_scale="annual",
    composite_stat="median"
)

In [19]:
# Build HLS Monthly Multiband Composite
hls_monthly_col = build_period_composites(
    collection=hls_col,
    bands=HLS_MERGED_BANDS,
    start_date=ee.Date(BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    temporal_scale="monthly",
    composite_stat="median"
)

In [20]:
# Reduce each annual image over all sites
hls_annual_site_fc = collection_to_site_timeseries(
    collection=hls_annual_col,
    sites_fc=sites_fc,
    bands=HLS_MERGED_BANDS,
    reducer=ee.Reducer.mean(),
    scale=30,
    tile_scale=4
)

In [21]:
# Reduce each annual image over all sites
hls_monthly_site_fc = collection_to_site_timeseries(
    collection=hls_monthly_col,
    sites_fc=sites_fc,
    bands=HLS_MERGED_BANDS,
    reducer=ee.Reducer.mean(),
    scale=30,
    tile_scale=4
)

In [ ]:
# Export Long ee.FeatureCollection to Drive as CSV
export_table_to_drive(
    collection=hls_annual_site_fc,
    description="tgbs_hls_annual_composite_all_bands_2014_2025_full",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_hls_annual_composite_all_bands_2014_2025_full",
    fileFormat="CSV"
)

In [ ]:
# Export Long ee.FeatureCollection to Drive as CSV
export_table_to_drive(
    collection=hls_monthly_site_fc,
    description="tgbs_hls_monthly_composite_all_bands_2014_2025_full",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_hls_monthly_composite_all_bands_2014_2025_full",
    fileFormat="CSV"
)

#### Build Sentinel-2 Annual and Monthly Composite Collections

In [5]:
# Build Sentinel-2 SR Source Collection
s2_col = get_s2_sr_collection(
    aoi=sites_fc,
    start_date=ee.Date(S2_BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    apply_water_masking=True
)

In [6]:
# Build HLS Annual Multiband Composite
s2_annual_col = build_period_composites(
    collection=s2_col,
    bands=S2_ALL_BANDS,
    start_date=ee.Date(S2_BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    temporal_scale="annual",
    composite_stat="median"
)

In [7]:
# Build HLS Monthly Multiband Composite
s2_monthly_col = build_period_composites(
    collection=s2_col,
    bands=S2_ALL_BANDS,
    start_date=ee.Date(S2_BASELINE_START),
    end_date=ee.Date(CURRENT_END),
    temporal_scale="monthly",
    composite_stat="median"
)

In [8]:
# Reduce each annual image over all sites
s2_annual_site_fc = collection_to_site_timeseries(
    collection=s2_annual_col,
    sites_fc=sites_fc,
    bands=S2_ALL_BANDS,
    reducer=ee.Reducer.mean(),
    scale=30,
    tile_scale=4
)

In [9]:
# Reduce each annual image over all sites
s2_monthly_site_fc = collection_to_site_timeseries(
    collection=s2_monthly_col,
    sites_fc=sites_fc,
    bands=S2_ALL_BANDS,
    reducer=ee.Reducer.mean(),
    scale=30,
    tile_scale=4
)

In [ ]:
# Export Long ee.FeatureCollection to Drive as CSV
export_table_to_drive(
    collection=s2_annual_site_fc,
    description="tgbs_s2_annual_composite_all_bands_2019_2025_full",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_s2_annual_composite_all_bands_2019_2025_full",
    fileFormat="CSV"
)

In [ ]:
# Export Long ee.FeatureCollection to Drive as CSV
export_table_to_drive(
    collection=s2_monthly_site_fc,
    description="tgbs_s2_monthly_composite_all_bands_2019_2025_full",
    folder=DRIVE_FOLDER,
    fileNamePrefix="tgbs_s2_monthly_composite_all_bands_2019_2025_full",
    fileFormat="CSV"
)